# unemployment hysteresis project

trying to see if shocks to the unemployment rate (UNRATE) stick around or fade out, using avg hourly earnings (AHETPI) and real GDP (GDPC1) as controls. quarterly data.

basically working through a bunch of time series models to compare - AR(1), MA(1), ARMA(1,4), ARIMA(0,1,0), DL(4), ADL(1,1), ARDL(4,3,3), VAR(5), VECM(r=1)

data is from FRED, csv is in the repo (merged_clean_data.csv)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
from statsmodels.regression.linear_model import OLS

# load the merged dataset
df = pd.read_csv('merged_clean_data.csv', parse_dates=['DATE'])
df.set_index('DATE', inplace=True)
df.index = pd.DatetimeIndex(df.index).to_period('Q')   # quarterly

UNRATE = df['UNRATE']
AHETPI = df['AHETPI']
GDPC1  = df['GDPC1']

print(f"Sample: {df.index[0]} to {df.index[-1]}  ({len(df)} quarters)")
df.describe().round(3)


## 1. AR(1)
$$UNRATE_t = c + \phi_1\, UNRATE_{t-1} + \varepsilon_t$$


In [ ]:
ar1 = ARIMA(UNRATE, order=(1, 0, 0)).fit()
print("AR(1) results")
print(ar1.summary().tables[1])
print("aic:", round(ar1.aic, 4), " bic:", round(ar1.bic, 4), " llf:", round(ar1.llf, 4))


## 2. MA(1)
$$UNRATE_t = c + \varepsilon_t + \theta_1\,\varepsilon_{t-1}$$


In [ ]:
ma1 = ARIMA(UNRATE, order=(0, 0, 1)).fit()
print("MA(1) results")
print(ma1.summary().tables[1])
print("aic:", round(ma1.aic, 4), " bic:", round(ma1.bic, 4), " llf:", round(ma1.llf, 4))


## 3. ARMA(1,4)
mixed model, just seeing if adding MA terms helps over plain AR(1)
$$UNRATE_t = c + \phi_1 UNRATE_{t-1} + \varepsilon_t + \theta_1\varepsilon_{t-1} + \cdots + \theta_4\varepsilon_{t-4}$$


In [ ]:
arma14 = ARIMA(UNRATE, order=(1, 0, 4)).fit()
print("ARMA(1,4) results")
print(arma14.summary().tables[1])
print("aic:", round(arma14.aic, 4), " bic:", round(arma14.bic, 4), " llf:", round(arma14.llf, 4))


## 4. ARIMA(0,1,0) - random walk on the first difference
this is the important one for the hysteresis question - if UNRATE is basically a random walk then shocks never go away
$$\Delta UNRATE_t = c + \varepsilon_t$$


In [ ]:
arima010 = ARIMA(UNRATE, order=(0, 1, 0)).fit()
print("ARIMA(0,1,0) results")
print(arima010.summary().tables[1])
print("aic:", round(arima010.aic, 4), " bic:", round(arima010.bic, 4), " llf:", round(arima010.llf, 4))


## 5. DL(4) - distributed lag
throwing in 4 lags of AHETPI and GDPC1, no lagged UNRATE yet
$$UNRATE_t = c + \sum_{j=0}^{4} \beta_j\, AHETPI_{t-j} + \sum_{j=0}^{4} \gamma_j\, GDPC1_{t-j} + \varepsilon_t$$


In [ ]:
def build_dl(endog, *exog_series, lags=4):
    # helper to build a distributed lag design matrix, nothing fancy
    frames = []
    for x in exog_series:
        name = x.name
        for j in range(lags + 1):
            col_name = f"{name}_L{j}" if j > 0 else name
            frames.append(x.shift(j).rename(col_name))
    X = pd.concat(frames, axis=1).dropna()
    y = endog.loc[X.index]
    return y, sm.add_constant(X)

y_dl, X_dl = build_dl(UNRATE, AHETPI, GDPC1, lags=4)
dl4 = OLS(y_dl, X_dl).fit(cov_type='HC1')

print("DL(4) results (HC1 robust SE)")
coef_dl = pd.DataFrame({
    'coef': dl4.params,
    'std err': dl4.bse,
    't': dl4.tvalues,
    'p': dl4.pvalues
}).round(5)
print(coef_dl.to_string())
print("R2:", round(dl4.rsquared, 4), " adj R2:", round(dl4.rsquared_adj, 4))
print("F-stat:", round(dl4.fvalue, 4), " (p=", round(dl4.f_pvalue, 4), ")")


## 6. ADL(1,1)
adding one lag of UNRATE itself now, plus current + 1 lag of each exogenous var
$$UNRATE_t = c + \phi_1 UNRATE_{t-1} + \beta_0 AHETPI_t + \beta_1 AHETPI_{t-1} + \gamma_0 GDPC1_t + \gamma_1 GDPC1_{t-1} + \varepsilon_t$$


In [ ]:
adl_data = pd.DataFrame({
    'UNRATE'   : UNRATE,
    'UNRATE_L1': UNRATE.shift(1),
    'AHETPI'   : AHETPI,
    'AHETPI_L1': AHETPI.shift(1),
    'GDPC1'    : GDPC1,
    'GDPC1_L1' : GDPC1.shift(1),
}).dropna()

y_adl = adl_data['UNRATE']
X_adl = sm.add_constant(adl_data.drop(columns='UNRATE'))

adl = OLS(y_adl, X_adl).fit(cov_type='HC1')

print("ADL(1,1) results (HC1 robust SE)")
coef_adl = pd.DataFrame({
    'coef': adl.params,
    'std err': adl.bse,
    't': adl.tvalues,
    'p': adl.pvalues
}).round(5)
print(coef_adl.to_string())
print("R2:", round(adl.rsquared, 4), " adj R2:", round(adl.rsquared_adj, 4))
print("F-stat:", round(adl.fvalue, 4), " (p=", round(adl.f_pvalue, 4), ")")


## 7. ARDL(4,3,3)
$$UNRATE_t = c + \sum_{j=1}^{4}\phi_j UNRATE_{t-j} + \sum_{j=0}^{3}\beta_j AHETPI_{t-j} + \sum_{j=0}^{3}\gamma_j GDPC1_{t-j} + \varepsilon_t$$


In [ ]:
p, q1, q2 = 4, 3, 3   # ARDL(4,3,3)

frames_ardl = {'const': pd.Series(1, index=UNRATE.index, name='const')}
for j in range(1, p + 1):
    frames_ardl[f'UNRATE_L{j}'] = UNRATE.shift(j)
for j in range(0, q1 + 1):
    lbl = 'AHETPI' if j == 0 else f'AHETPI_L{j}'
    frames_ardl[lbl] = AHETPI.shift(j)
for j in range(0, q2 + 1):
    lbl = 'GDPC1' if j == 0 else f'GDPC1_L{j}'
    frames_ardl[lbl] = GDPC1.shift(j)

ardl_df = pd.DataFrame(frames_ardl).dropna()
y_ardl  = UNRATE.loc[ardl_df.index]
X_ardl  = ardl_df

ardl = OLS(y_ardl, X_ardl).fit(cov_type='HC1')

print("ARDL(4,3,3) results (HC1 robust SE)")
coef_ardl = pd.DataFrame({
    'coef': ardl.params,
    'std err': ardl.bse,
    't': ardl.tvalues,
    'p': ardl.pvalues
}).round(5)
print(coef_ardl.to_string())
print("R2:", round(ardl.rsquared, 4), " adj R2:", round(ardl.rsquared_adj, 4))
print("F-stat:", round(ardl.fvalue, 4), " (p=", round(ardl.f_pvalue, 4), ")")

# long run multipliers, LRM = sum(beta) / (1 - sum(phi))
phi_sum  = sum(ardl.params.get(f'UNRATE_L{j}', 0) for j in range(1, p + 1))
beta_sum = sum(ardl.params.get(f'AHETPI_L{j}' if j > 0 else 'AHETPI', 0) for j in range(0, q1 + 1))
gamma_sum= sum(ardl.params.get(f'GDPC1_L{j}'  if j > 0 else 'GDPC1',  0) for j in range(0, q2 + 1))

denom = 1 - phi_sum
print("\nlong run multipliers")
print("sum phi (AR part):", round(phi_sum, 5))
print("LRM AHETPI -> UNRATE:", round(beta_sum / denom, 5))
print("LRM GDPC1 -> UNRATE:", round(gamma_sum / denom, 5))


## 8. VAR(5)
system of [UNRATE, AHETPI, GDPC1], 5 lags (fixed, not selected by IC)


In [ ]:
var_data = df[['UNRATE', 'AHETPI', 'GDPC1']].dropna()
var_model = VAR(var_data)
var5 = var_model.fit(maxlags=5, ic=None)   # forcing lag=5

var_coef = var5.params
var_bse  = var5.stderr
var_tval = var5.tvalues
var_pval = var5.pvalues

for eq_name in ['UNRATE', 'AHETPI', 'GDPC1']:
    print(f"\nequation: {eq_name}")
    eq_df = pd.DataFrame({
        'coef': var_coef[eq_name],
        'std err': var_bse[eq_name],
        't': var_tval[eq_name],
        'p': var_pval[eq_name]
    }).round(5)
    print(eq_df.to_string())

print("\nVAR(5) model stats")
print("aic:", round(var5.aic, 4), " bic:", round(var5.bic, 4), " fpe:", round(var5.fpe, 6), " hqic:", round(var5.hqic, 4))


## 9. VECM (r=1)
cointegration rank 1, same 3-variable system. this one gives us the long run equilibrium relationship plus how fast UNRATE snaps back to it (the alpha coefficient basically IS the hysteresis speed)


In [ ]:
vecm_data = df[['UNRATE', 'AHETPI', 'GDPC1']].dropna()

# johansen test first just to check the rank makes sense
print("Johansen cointegration rank test (det_order=0, k_ar_diff=4)")
rank_res = select_coint_rank(vecm_data, det_order=0, k_ar_diff=4, method='trace')
print(rank_res.summary())

# fitting with r=1 like we said we would
vecm_model = VECM(vecm_data, k_ar_diff=4, coint_rank=1, deterministic='co')
vecm_fit   = vecm_model.fit()

print("\nVECM (r=1, k_ar_diff=4) summary")
print(vecm_fit.summary())

print("\nalpha (adjustment coefficients)")
alpha_df = pd.DataFrame(
    vecm_fit.alpha,
    index  = vecm_data.columns,
    columns= [f'EC{i+1}' for i in range(1)]
).round(6)
print(alpha_df.to_string())

print("\nbeta (cointegrating vector, normalised to UNRATE)")
beta_df = pd.DataFrame(
    vecm_fit.beta,
    index  = vecm_data.columns,
    columns= [f'CV{i+1}' for i in range(1)]
).round(6)
print(beta_df.to_string())

print("\ngamma (short run coefficient matrices, one per lag)")
for i, gamma in enumerate(vecm_fit.gamma.reshape(-1, 3, 3), start=1):
    g_df = pd.DataFrame(
        gamma,
        index  = [f'd{v}' for v in vecm_data.columns],
        columns= vecm_data.columns
    ).round(6)
    print(f"\nlag {i}:")
    print(g_df.to_string())


## summary table
just lining up AIC/BIC (and R2 for the OLS-based ones) so it's easy to eyeball


In [ ]:
summary = pd.DataFrame([
    {'Model': 'AR(1)',         'AIC': round(ar1.aic,   2), 'BIC': round(ar1.bic,   2), 'LogLik': round(ar1.llf,   4)},
    {'Model': 'MA(1)',         'AIC': round(ma1.aic,   2), 'BIC': round(ma1.bic,   2), 'LogLik': round(ma1.llf,   4)},
    {'Model': 'ARMA(1,4)',     'AIC': round(arma14.aic,2), 'BIC': round(arma14.bic,2), 'LogLik': round(arma14.llf,4)},
    {'Model': 'ARIMA(0,1,0)',  'AIC': round(arima010.aic,2),'BIC': round(arima010.bic,2),'LogLik': round(arima010.llf,4)},
    {'Model': 'DL(4)',         'AIC': round(dl4.aic,   2), 'BIC': round(dl4.bic,   2), 'R2': round(dl4.rsquared,4)},
    {'Model': 'ADL(1,1)',      'AIC': round(adl.aic,   2), 'BIC': round(adl.bic,   2), 'R2': round(adl.rsquared,4)},
    {'Model': 'ARDL(4,3,3)',   'AIC': round(ardl.aic,  2), 'BIC': round(ardl.bic,  2), 'R2': round(ardl.rsquared,4)},
    {'Model': 'VAR(5)',        'AIC': round(var5.aic,  2), 'BIC': round(var5.bic,  2), 'LogLik': round(var5.llf, 4)},
])
print("model comparison")
print(summary.fillna('-').to_string(index=False))


---
## plots

quick matplotlib plots to actually look at this stuff instead of just staring at coefficient tables


### the raw series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(df.index.to_timestamp(), UNRATE, color='tab:blue')
axes[0].set_ylabel('UNRATE')
axes[0].set_title('unemployment rate, avg hourly earnings, and real gdp over time')

axes[1].plot(df.index.to_timestamp(), AHETPI, color='tab:orange')
axes[1].set_ylabel('AHETPI')

axes[2].plot(df.index.to_timestamp(), GDPC1, color='tab:green')
axes[2].set_ylabel('GDPC1')
axes[2].set_xlabel('date')

plt.tight_layout()
plt.show()


### AR(1) fitted vs actual
just to see how well a simple AR(1) tracks the real thing

In [ ]:
fitted_ar1 = ar1.fittedvalues

plt.figure(figsize=(10, 4))
plt.plot(df.index.to_timestamp(), UNRATE, label='actual', color='black', linewidth=1)
plt.plot(df.index.to_timestamp(), fitted_ar1, label='AR(1) fitted', color='tab:red', linewidth=1)
plt.title('UNRATE: actual vs AR(1) fitted')
plt.xlabel('date')
plt.ylabel('unemployment rate')
plt.legend()
plt.tight_layout()
plt.show()


### residuals from the random walk model
this is the one that matters for the hysteresis question - if these look like white noise with no structure, that's consistent with shocks being permanent

In [ ]:
resid_rw = arima010.resid

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(resid_rw.index.to_timestamp(), resid_rw, color='tab:purple', linewidth=1)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('ARIMA(0,1,0) residuals over time')
axes[0].set_xlabel('date')

axes[1].hist(resid_rw.dropna(), bins=25, color='tab:purple', edgecolor='white')
axes[1].set_title('residual distribution')

plt.tight_layout()
plt.show()


### comparing AIC across models
lower is better. left out DL/ADL/ARDL/VECM here since AIC isn't directly comparable once you change how the likelihood is set up (differenced vs level data etc), so this is just the ones that are apples-to-apples

In [ ]:
aic_compare = summary.dropna(subset=['AIC'])[['Model', 'AIC']]

plt.figure(figsize=(8, 4))
plt.bar(aic_compare['Model'], aic_compare['AIC'], color='tab:blue')
plt.ylabel('AIC')
plt.title('AIC by model (lower = better fit)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


### VAR(5) impulse response
how UNRATE responds over time to a one-time shock in each variable - the slow decay (or lack of it) is basically what hysteresis looks like in this framework

In [ ]:
irf = var5.irf(12)
irf.plot(impulse='UNRATE')
plt.tight_layout()
plt.show()
